In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")


Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [6]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

store_summary = (
    df.groupBy("category")
    .agg(
        _min("amount").alias("najniższa kwota"),
        _max("amount").alias("największa kwota"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("category")
)
store_summary.show()

+-----------+---------------+----------------+----------+
|   category|najniższa kwota|największa kwota|  suma_PLN|
+-----------+---------------+----------------+----------+
|elektronika|            9.0|          9999.0|1520770.69|
|    książki|            5.0|         9107.25| 851382.08|
|     odzież|            5.0|         9696.63| 849877.55|
|    żywność|            5.0|         6916.92| 789514.43|
+-----------+---------------+----------------+----------+



In [28]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)


(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [16]:
from pyspark.sql.functions import window

half_hourly = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)


(
    half_hourly
    .select(
        "store",
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+--------+-------------------+-------------------+---------+---------+
|store   |od                 |do                 |liczba_tx|suma_PLN |
+--------+-------------------+-------------------+---------+---------+
|Gdańsk  |2026-04-12 08:00:00|2026-04-12 08:30:00|252      |93391.22 |
|Kraków  |2026-04-12 08:00:00|2026-04-12 08:30:00|289      |117786.42|
|Warszawa|2026-04-12 08:00:00|2026-04-12 08:30:00|275      |88441.58 |
|Wrocław |2026-04-12 08:00:00|2026-04-12 08:30:00|296      |111540.59|
|Gdańsk  |2026-04-12 08:30:00|2026-04-12 09:00:00|514      |209187.85|
|Kraków  |2026-04-12 08:30:00|2026-04-12 09:00:00|532      |223541.41|
|Warszawa|2026-04-12 08:30:00|2026-04-12 09:00:00|490      |182435.06|
|Wrocław |2026-04-12 08:30:00|2026-04-12 09:00:00|502      |215587.17|
|Gdańsk  |2026-04-12 09:00:00|2026-04-12 09:30:00|619      |253364.95|
|Kraków  |2026-04-12 09:00:00|2026-04-12 09:30:00|590      |224358.03|
|Warszawa|2026-04-12 09:00:00|2026-04-12 09:30:00|584      |214573.66|
|Wrocł

In [17]:
from pyspark.sql.functions import desc

df_krk = df.filter(col("store")=="Kraków")

hourly = (
    df_krk.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("suma_PLN"))
)


(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532      |201259.26|
+-------------------+-------------------+---------+---------+



In [18]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [ ]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Jest więcej okien czasowych, bo te na siebie nachodzą. W przypadku sliding okna po prostu tworzone są częściej, więc w tym samym odcinku czasu jest ich więcej od tumbling

In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ:
# Między 9:00 a 10:00 (zadanie 3.1.) nastąpiło 4661 transakcji. 

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ:
""" W pierwszym wypadku agregacja nastąpi tylko po sklepach (brak więc wartości zagregowanych dla konkretnych godzin), a w drugim przypadku agregacja następi odpowiednio po godzinach zegarowych
    oraz po sklepach (jeden wiersz będzie mówił o sprzedaży w danej godzinie dla dengo sklepu)."""

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ:
# Taka transakcja zostanie zawarta w oknie 9:00-10:00 oraz 9:30-10:30, a więc w dwóch oknach.

In [22]:
# Praca domowa 1

from pyspark.sql.functions import avg, round as _round, asc

df_gd = df.filter(col("store")=="Gdańsk")

hourly = (
    df_gd.groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_kwota")
    )
    .orderBy(asc("srednia_kwota"))
)


(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_kwota"
    )
    .show(truncate=False)
)

# Najniższa średnia kwota transakcji występuje dla okna 8:00-9:00 

+-------------------+-------------------+-------------+
|od                 |do                 |srednia_kwota|
+-------------------+-------------------+-------------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01       |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92       |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91       |
+-------------------+-------------------+-------------+



In [27]:
# Praca domowa 2

from pyspark.sql.functions import count as _count

half_hourly = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        _count("tx_id").alias("liczba_tx")
    )
)

half_hourly = half_hourly.where(col("window.start") == "2026-04-12 9:00:00")

(
    half_hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_tx"
    )
    .show(truncate=False)
)


+-------------------+-------------------+-----------+---------+
|od                 |do                 |category   |liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|611      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|odzież     |605      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|książki    |622      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|żywność    |567      |
+-------------------+-------------------+-----------+---------+



In [31]:
# Praca domowa 3
# Za szczyt transakcji przyjmuję ich liczbę, a nie sumaryczną kwotę.

from pyspark.sql.functions import window, count, desc

quarterly = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(desc("liczba_tx"))
)


(
    quarterly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
    )
    .show(truncate=False)
)

# Najwięcej transakcji miało miejsce między 9:15 a 9:30

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+

